# A corrected sampler in 281 steps, and a probability flow that flies off

Companion notebook to T. Namba, *Learned scores on the critically damped Langevin flow: a corrected sampler in 281 steps, and a metric-dependent fragility of the probability flow*, Zenodo (2026), [doi:10.5281/zenodo.23899157](https://doi.org/10.5281/zenodo.23899157). Code: [cld-learned-score-samplers](https://github.com/neginyan/cld-learned-score-samplers).

It runs on a CPU in about **3 minutes** and uses one of the 28 trained score networks stored in the repository; nothing is trained here.

**Setting.** Critically damped Langevin (CLD) flow in position–velocity phase space, a four-dimensional Gaussian mixture (eight modes on a circle), and an isotropic noise schedule specified by its marginals, σ(t) = σ_min e^{rt} with σ_min = 0.005 and r = 1. *The standard CLD, with velocity-only noise given by its SDE, does not show the failures shown here.*

1. **Same network, same Brownian path, same budget.** The corrected reverse SDE with a Heun-type step at 281 steps, against Euler–Maruyama at 281 steps and at 562 steps (the same number of score evaluations).
2. **Why the probability-flow ODE breaks.** A broken run is broken by a handful of tail trajectories that fly far. We re-run exactly those trajectories with the learned and with the exact score.

In [ ]:
# Get the code and the trained networks (about 20 MB)
import os, sys
if not os.path.isdir('cld-learned-score-samplers'):
    !git clone -q --depth 1 https://github.com/neginyan/cld-learned-score-samplers
os.chdir('cld-learned-score-samplers'); sys.path.insert(0, 'src')
import json, time
import numpy as np
import matplotlib.pyplot as plt
import g2samplers as g
m, ls, p, y0, ref = g.setup('I', 1, 0)     # metric P = I, network 1, sampling seed 0, N = 20 000 samples
print('ready:', len(y0), 'samples from the prior, metric P = I (kappa_P = 2 > r = 1)')

## 1. The corrected SDE at a coarse step

The corrected reverse SDE has the drift `f = B_P y - 0.5 (G + G_plus) s_hat` and the noise `N(0, h G_plus)`. The Heun-type step uses the **same** Gaussian increment in both stages:

```diff
-y_next = y - h * f(y, t) + dW                               # Euler-Maruyama
+y_pred = y - h * f(y, t) + dW
+y_next = y - 0.5 * h * (f(y, t) + f(y_pred, t - h)) + dW    # Heun-type
```

`L` is the log ratio of the within-mode position variance to its target (0 is exact). The three runs below share one Brownian path. About 2 minutes.

In [ ]:
runs = [('Heun-type, 281 steps', g.heun, 281, 16),
        ('Euler-Maruyama, 281 steps', g.em, 281, 16),
        ('Euler-Maruyama, 562 steps (same cost as Heun 281)', g.em, 562, 8)]
out = {}
for name, fn, K, sub in runs:
    t0 = time.time(); zY = fn(ls, p, y0, K, sub, 0); st = g.summary(m, ref, zY)
    out[name] = (m.to_Z(zY), st)
    print(f"{name:52s} L = {st['L']:+.3f}   outliers = {st['n_bad']}   ({time.time() - t0:.0f} s)")
print('\nPaper, Table 1 (median over 8 runs, P = I): Heun 281 steps 0.056; EM 281 steps 0.416; EM 562 steps 0.246')

In [ ]:
# Deviations from the nearest mode centre, pooled over the eight modes (positions x1, x2)
mu = m.mu[:, [0, 2]]; s_target = np.sqrt(0.08**2 + 0.005**2)
def dev(z):
    x = z[np.isfinite(z).all(1)][:, [0, 2]]
    return x - mu[np.argmin(((x[:, None] - mu[None])**2).sum(2), 1)]
names = list(out); cols = ['#1c5cab', '#d1495b', '#e8a33d']
fig, axs = plt.subplots(1, 3, figsize=(14, 4.4), gridspec_kw={'width_ratios': [1, 1, 1.25]})
th = np.linspace(0, 2 * np.pi, 200)
for ax, name, c in zip(axs[:2], names[:2], cols):
    d = dev(out[name][0]); ax.scatter(d[:, 0], d[:, 1], s=1, alpha=0.25, color=c)
    for k in (1, 2): ax.plot(k * s_target * np.cos(th), k * s_target * np.sin(th), color='k', lw=1)
    ax.set_title(f"{name}\nL = {out[name][1]['L']:+.3f}", fontsize=10); ax.set_aspect('equal')
    ax.set_xlim(-0.4, 0.4); ax.set_ylim(-0.4, 0.4); ax.set_xlabel('x1 - mode centre')
axs[0].set_ylabel('x2 - mode centre')
r = np.linspace(0, 0.45, 300); axs[2].plot(r, r / s_target**2 * np.exp(-r**2 / (2 * s_target**2)), 'k', lw=2, label='target')
for name, c in zip(names, cols):
    rr = np.sqrt((dev(out[name][0])**2).sum(1))
    axs[2].hist(rr, bins=np.linspace(0, 0.45, 60), density=True, histtype='step', lw=1.6, color=c, label=f"{name.split(' (')[0]}, L = {out[name][1]['L']:+.2f}")
axs[2].set_xlabel('distance from the mode centre'); axs[2].set_ylabel('density'); axs[2].legend(fontsize=8)
axs[2].set_title('Radial profile of the generated modes', fontsize=10)
fig.suptitle('Same learned score, same Brownian path (black circles: 1 and 2 standard deviations of the target)', fontsize=11, y=1.0)
plt.tight_layout(); plt.show()

The Heun-type sampler lands on the target at 281 steps. Euler–Maruyama spreads every mode, and even at the same cost (562 steps) it does not catch up. In the paper this holds in all four metrics tested (median |L| ≤ 0.083 at 281 steps; Euler–Maruyama 0.42–0.55); at 140 steps the Heun-type sampler no longer suffices.

## 2. The probability-flow ODE: a few trajectories fly far

With the same network, the probability-flow ODE (4496 steps) gives L ≈ 90 for this seed. The ODE moves every point independently, so we can re-run only the trajectories that ended far from every mode in the stored run, together with 300 other points, once with the **learned** and once with the **exact** score. About 20 seconds.

In [ ]:
far = json.load(open('results/B9/samp_I_n1_s0.json'))['levels'][-1]['ode']['far_idx']
rng = np.random.default_rng(0); others = rng.choice(np.setdiff1d(np.arange(len(y0)), far), 300, replace=False)
sel = np.concatenate([far, others]); exact = m.score()
zl = m.to_Z(g.ode(ls, p, y0[sel])); ze = m.to_Z(g.ode(exact, p, y0[sel]))
def dist(z):
    d = np.full(len(z), np.inf); f = np.isfinite(z).all(1)
    d[f] = np.sqrt(((z[f][:, None, [0, 2]] - mu[None])**2).sum(2)).min(1); return d
dl, de = dist(zl), dist(ze)
print('the', len(far), 'trajectories that broke the stored run (of 20 000):')
for i in range(len(far)): print(f'  start {far[i]:5d}: distance to the nearest mode   learned {dl[i]:.3g}   exact {de[i]:.3g}')
print(f'300 other points farther than 0.5: learned score {(dl[len(far):] > 0.5).sum()}, exact score {(de[len(far):] > 0.5).sum()}')
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(de[len(far):], np.clip(dl[len(far):], 1e-3, 1e6), s=8, color='#9aa5b1', label='300 other points')
ax.scatter(de[:len(far)], np.clip(dl[:len(far)], 1e-3, 1e6), s=40, color='#d1495b', label='the trajectories that broke the run')
ax.set_xscale('log'); ax.set_yscale('log'); ax.plot([1e-3, 1e6], [1e-3, 1e6], 'k:', lw=0.8)
ax.axhline(0.5, color='#d1495b', lw=0.8, ls='--'); ax.set_xlim(1e-3, 1); ax.set_ylim(1e-3, 3e6)
ax.set_xlabel('distance to the nearest mode, exact score'); ax.set_ylabel('same start, learned score (clipped at 1e6)')
ax.legend(fontsize=9, loc='upper left'); ax.set_title('Probability-flow ODE, P = I, network 1'); plt.tight_layout(); plt.show()

With the exact score the same starting points end near the data. With the learned score a handful of tail trajectories fly far, one of them by about 10^20, and that alone makes |L| large: a broken run contains at most 0.12% outliers. In the paper these trajectories start in the tail of the prior, pick up most of their deviation while σ is large, and jump at the end. The learned probability-flow ODE broke for 10 of 12 networks with P = I and for 3 of 16 with metrics of κ_P ≤ 0; thirteen of nineteen pre-registered predictions failed, and all of them are reported in the paper.

*All numbers here are recomputed from the stored network; `verification/run_all.py` in the repository checks every number of the paper.*